# 03 — From probability to decisionNotebook 02 produced a calibrated probability for every booking in the held-out period.A probability is not a decision. This notebook turns them into the two things a revenuemanager can act on:**How many rooms can be sold above capacity tonight, and what is the risk?****Which bookings are worth a phone call?**Everything here runs on `outputs/test_predictions.csv` — the five months the model neversaw during training. Nothing is fitted in this notebook.

In [ ]:
import syssys.path.append("..")from pathlib import Pathimport numpy as npimport pandas as pdfrom src.features import TARGET, load_bookingsOUTPUTS = Path("..") / "outputs"predictions = pd.read_csv(OUTPUTS / "test_predictions.csv", parse_dates=["arrival_date"])predictions = predictions[predictions["nights"] > 0].copy()print(f"{len(predictions):,} bookings, arrivals "      f"{predictions['arrival_date'].min().date()} to {predictions['arrival_date'].max().date()}")

## 1. How large are these hotels?Neither the dataset nor the source paper states a room count, and without one there is nosuch thing as overbooking.It can be recovered. Occupancy on any night cannot exceed the number of rooms, so if thehotels ever sell out, the distribution of nightly occupancy will stop at a hard ceiling —and that ceiling is the capacity.Each realised booking is expanded into one row per night of stay, across the full historyrather than the test period, so the estimate uses every night on record.

In [ ]:
def expand_to_nights(df, night_column="arrival_date"):    """One row per booking per night of stay."""    stretched = df.loc[df.index.repeat(df["nights"])].copy()    stretched["night_offset"] = stretched.groupby(level=0).cumcount()    stretched["night"] = (stretched[night_column]                          + pd.to_timedelta(stretched["night_offset"], unit="D"))    return stretchedhistory = load_bookings()history["nights"] = (history["stays_in_weekend_nights"] + history["stays_in_week_nights"])realised = history[(history["nights"] > 0) & (history[TARGET] == 0)]occupancy = (expand_to_nights(realised)             .groupby(["hotel", "night"]).size().rename("rooms_occupied").reset_index())print(occupancy.groupby("hotel")["rooms_occupied"]      .describe(percentiles=[0.5, 0.95, 0.99]).round(1).to_string())

In [ ]:
for hotel, nights in occupancy.groupby("hotel"):    highest = nights["rooms_occupied"].value_counts().sort_index(ascending=False).head(6)    print(f"{hotel} — nights spent at each of the highest occupancy levels")    print(highest.to_string(), "\n")CAPACITY = occupancy.groupby("hotel")["rooms_occupied"].max().to_dict()print("inferred capacity:", CAPACITY)

The pile-up is the evidence. The city hotel spends dozens of nights between 219 and 226rooms and never goes beyond 226; the resort clusters around 179–184 and stops at 187. Thatis the shape of a constraint, not of demand tapering off.Two caveats, because this is an inference and not a fact:- The maximum observed is a **lower bound**. If a hotel never quite filled, the true room  count is slightly higher and every recommendation below is correspondingly conservative.- Rooms out of service for maintenance would depress the ceiling in the same way.The resort's 187 rests on a single night, with the next level at 185. A ceiling of 185would be a defensible reading too. The difference propagates directly into how many roomsmay be oversold, which is why it is stated here rather than assumed away.

## 2. Forecasting a nightFor a given night, each booking either arrives or does not. The model gives each one acancellation probability, so the number of arrivals is a sum of independent Bernoullitrials with *different* probabilities — a Poisson-binomial distribution.Its first two moments are simple:$$\mu = \sum_i (1 - p_i) \qquad \sigma^2 = \sum_i p_i (1 - p_i)$$With a few hundred bookings a night, a normal approximation to the total issound. The mean matters less than the spread: **the uncertainty is what determines howaggressively a hotel may oversell.** Two nights expecting 200 arrivals are not equallysafe if one is built from confident predictions and the other from coin flips.

In [ ]:
nightly = expand_to_nights(predictions)nightly["arrives"] = 1 - nightly["cancel_probability"]forecast = (nightly.groupby(["hotel", "night"])            .agg(bookings_held=("arrives", "size"),                 expected_arrivals=("arrives", "sum"),                 variance=("cancel_probability", lambda p: (p * (1 - p)).sum()),                 actual_arrivals=(TARGET, lambda s: int((s == 0).sum())),                 mean_adr=("adr", "mean"))            .reset_index())forecast["uncertainty"] = np.sqrt(forecast["variance"])forecast["capacity"] = forecast["hotel"].map(CAPACITY)# Nights at the edges of the window are incomplete: a stay beginning in March and running# into April contributes rooms the test set does not contain, and the same applies after# 31 August. Only fully covered nights are analysed.COMPLETE = (forecast["night"] >= "2017-05-01") & (forecast["night"] <= "2017-08-31")forecast = forecast[COMPLETE].reset_index(drop=True)print(f"{len(forecast)} nights analysed, "      f"{forecast['night'].min().date()} to {forecast['night'].max().date()}")print(forecast.groupby("hotel")[["bookings_held", "expected_arrivals",                                 "actual_arrivals", "uncertainty"]].mean().round(1).to_string())

Both hotels hold far more bookings than they have rooms — the city hotel averages over300 bookings against 226 rooms. Overbooking is not a strategy here; it is the existingstate of the business. The only question is how much of it is deliberate.The average uncertainty of six to seven rooms is the number that makes the rest of thisnotebook possible. It says the nightly arrival count is predictable to within roughly adozen rooms either side, which is a tight enough band to act on.

In [ ]:
print("occupancy forecast accuracy over the analysed nights")accuracy = forecast.groupby("hotel").apply(    lambda g: pd.Series({        "expected": g["expected_arrivals"].sum(),        "actual": g["actual_arrivals"].sum(),        "error_%": round(100 * (g["expected_arrivals"].sum() - g["actual_arrivals"].sum())                         / g["actual_arrivals"].sum(), 1)}),    include_groups=False).round(0)print(accuracy.to_string())

Both hotels are over-predicted, the resort more than the city — the same asymmetry found innotebook 02, arriving here in the units that matter. Over-predicting arrivals means themodel expects a fuller hotel than materialises, so every recommendation below **errstowards caution**. Rooms are left unsold rather than guests left standing.That is the right direction for the error to point, but it is luck rather than design, andit should not be presented as a safety feature.

## 3. How many rooms can be sold above capacity?With arrivals distributed around $\mu$ with spread $\sigma$, the hotel picks a tolerance —how often is it willing to be unable to house someone — and sells up to the correspondingquantile.$$\text{rooms available} = \left\lfloor C - (\mu + z\sigma) \right\rfloor$$A 5% tolerance sets $z = 1.645$. This is a business decision, not a statistical one: walkinga guest costs an alternative room, compensation, and a review. The model supplies thedistribution; management supplies the appetite.

In [ ]:
TOLERANCES = {"1%": 2.326, "5%": 1.645, "10%": 1.282, "20%": 0.842}def recommend(z):    return np.floor(forecast["capacity"]                    - (forecast["expected_arrivals"] + z * forecast["uncertainty"])).clip(lower=0)def backtest(extra_rooms):    """Assume every extra room sells and that guest arrives — the pessimistic case."""    overflow = (forecast["actual_arrivals"] + extra_rooms - forecast["capacity"]).clip(lower=0)    return pd.Series({        "extra room-nights": int(extra_rooms.sum()),        "revenue (EUR)": round((extra_rooms * forecast["mean_adr"]).sum()),        "nights with a walk": int((overflow > 0).sum()),        "walk rate %": round(100 * (overflow > 0).mean(), 1),        "guests walked": int(overflow.sum())})sweep = pd.DataFrame({label: backtest(recommend(z)) for label, z in TOLERANCES.items()}).Tsweep.index.name = "tolerance"print(sweep.to_string())

Read the 5% row against its own target. The policy was built to walk a guest on at most 5%of nights; over the held-out period it did so on **3.7%**. The tolerance behaves asspecified, which is the practical proof that the calibration work in notebook 02 was worthdoing — an uncalibrated model would not land near its own stated risk.The backtest is deliberately pessimistic: it assumes every oversold room is taken *and*that guest turns up. In practice some of the extra rooms would go unsold.The scale is modest — a few hundred room-nights over four months, worth tens of thousandsof euros. That is the honest answer for two hotels already running close to full. Themethod matters more than the total: it is a rule that adapts nightly to how confident themodel is, instead of a fixed percentage applied blindly.

In [ ]:
CHOSEN = "5%"forecast["rooms_to_release"] = recommend(TOLERANCES[CHOSEN])forecast["projected_occupancy_%"] = (100 * forecast["expected_arrivals"]                                     / forecast["capacity"]).round(1)busiest = forecast.nlargest(8, "expected_arrivals")[    ["hotel", "night", "bookings_held", "expected_arrivals", "uncertainty",     "capacity", "rooms_to_release", "actual_arrivals"]]print(busiest.round(1).to_string(index=False))

This table is the working output. On the busiest nights the recommendation is zero — thehotel is genuinely full and there is nothing to release. That is the system declining totake a risk, and it is as important as the nights where it says yes.

## 4. Which bookings are worth a phone call?A confirmation call takes a few minutes and recovers only a fraction of the bookings itreaches. It has to be aimed.Ranking by probability alone sends staff after cheap one-night stays. Ranking by valuealone sends them after expensive bookings that were never going to cancel. The quantityworth ranking on is the product — **expected loss**.

In [ ]:
bookings_ranked = pd.read_csv(OUTPUTS / "test_predictions.csv", parse_dates=["arrival_date"])bookings_ranked["expected_loss"] = (bookings_ranked["cancel_probability"]                                    * bookings_ranked["booking_value"])total_exposure = bookings_ranked["expected_loss"].sum()rows = []for n in (200, 500, 1000, 2000):    top = bookings_ranked.nlargest(n, "expected_loss")    rows.append({"calls": n,                 "share of bookings %": round(100 * n / len(bookings_ranked), 2),                 "share of exposure %": round(100 * top["expected_loss"].sum() / total_exposure, 1),                 "actual cancel rate %": round(100 * top[TARGET].mean(), 1)})print(f"total exposure over the period: EUR {total_exposure:,.0f}\n")print(pd.DataFrame(rows).set_index("calls").to_string())print(f"\nbaseline cancellation rate: {100 * bookings_ranked[TARGET].mean():.1f}%")

The top 1,000 bookings — under 4% of the period — carry **20.5% of the exposure**, and**75.4% of them did cancel**, against a 41.2% baseline.That is the number to give an operations manager. Not an AUC: *call these thousand, threein four are leaving, and you are covering a fifth of the money at stake.*The list also answers the question that follows immediately. At roughly eight calls anhour, a thousand calls is around three staff-weeks across four months — a few hours a day.Whether that pays depends on how many bookings a call actually saves, which this datasetcannot answer. It would need an experiment: call a random half of the list, leave theother half, compare. Worth stating as the obvious next step rather than assuming thecalls work.

In [ ]:
call_list = (bookings_ranked.nlargest(1000, "expected_loss")             [["hotel", "arrival_date", "market_segment", "country", "lead_time",               "nights", "adr", "booking_value", "cancel_probability", "expected_loss",               "deposit_type", "customer_type"]]             .sort_values(["arrival_date", "expected_loss"], ascending=[True, False]))print(call_list.groupby("market_segment").agg(    calls=("expected_loss", "size"),    exposure=("expected_loss", "sum"),    mean_risk=("cancel_probability", "mean")).round(2).sort_values("exposure", ascending=False).to_string())

The list concentrates where notebook 01 and the original Power BI report both pointed:the segments that book far ahead, pay nothing up front, and cancel most.

## 5. Files for the reportThree tables, one per grain, for the Power BI page to consume directly.

In [ ]:
nightly_out = forecast[[    "hotel", "night", "bookings_held", "expected_arrivals", "uncertainty", "capacity",    "rooms_to_release", "projected_occupancy_%", "actual_arrivals", "mean_adr"]].copy()nightly_out["expected_cancellations"] = (nightly_out["bookings_held"]                                         - nightly_out["expected_arrivals"]).round(1)nightly_out["expected_arrivals"] = nightly_out["expected_arrivals"].round(1)nightly_out["uncertainty"] = nightly_out["uncertainty"].round(2)
nightly_out["mean_adr"] = nightly_out["mean_adr"].round(2)nightly_out.to_csv(OUTPUTS / "nightly_forecast.csv", index=False)call_list.to_csv(OUTPUTS / "call_list.csv", index=False)monthly = (bookings_ranked           .groupby([bookings_ranked["arrival_date"].dt.to_period("M").astype(str), "hotel"])           .agg(bookings=("cancel_probability", "size"),                predicted_cancellations=("cancel_probability", "sum"),                actual_cancellations=(TARGET, "sum"),                revenue_at_risk=("revenue_at_risk", "sum"))           .round(1).reset_index().rename(columns={"arrival_date": "month"}))
monthly.insert(1, "month_start", pd.to_datetime(monthly["month"] + "-01"))monthly.to_csv(OUTPUTS / "monthly_accuracy.csv", index=False)for name in ("nightly_forecast.csv", "call_list.csv", "monthly_accuracy.csv",             "test_predictions.csv"):    print(f"  {name:<26} {len(pd.read_csv(OUTPUTS / name)):>7,} rows")print()print(monthly.to_string(index=False))

## Summary| Question | Answer ||---|---|| How big are the hotels? | 226 and 187 rooms, inferred from the occupancy ceiling || How many arrive tonight? | Poisson-binomial: mean, and a spread of 6–7 rooms || How many rooms can be released? | 641 room-nights over four months at a 5% tolerance || Did the tolerance hold? | Yes — a walk on 3.7% of nights against a 5% budget || Who should be called? | 1,000 bookings covering 20.5% of exposure, 75.4% of which cancelled |**Limitations carried forward.** Capacity is inferred, not given. The walk backtest assumesevery released room sells and that guest arrives. Whether a confirmation call saves abooking is untested and would need an experiment. And the resort's arrivals remainover-predicted, which makes its recommendations the more conservative of the two.The Power BI page is built on `nightly_forecast.csv`, `call_list.csv` and`monthly_accuracy.csv`.